# Quantum feature reduction basics

This notebook prepares the four-feature input for the preliminary quantum experiment. It does not construct or train a QSVM. The reusable reduction logic lives in `src/quantum_features.py`.

## Why use only four features?

The preliminary experiment uses four features and four qubits so that the quantum circuit and kernel matrix remain tractable for a midterm study. Reducing the feature count reduces the number of qubits required by the feature map, but it does not by itself make the full QSVM training set small.

Feature selection is supervised and is fitted only on the training partition. The held-out test partition is transformed with the frozen selected columns and cannot influence the selection.

## What is a qubit?

A qubit is the basic unit of quantum information. Unlike a classical bit, which is either 0 or 1, a qubit can be in a superposition of basis states. Measurement produces a classical outcome. Four qubits provide a compact quantum register for the four selected input features.

## What does a quantum feature map do?

A quantum feature map is a parameterized circuit that encodes a classical vector into a quantum state. For this project, the eventual four-dimensional vector will be encoded by a current function-based `zz_feature_map` using four qubits. The feature map can also introduce interactions between encoded features.

## What does a quantum kernel represent?

A quantum kernel measures the similarity between two encoded quantum states. The fidelity kernel used in the planned experiment is

$$K(x, y) = |\langle \phi(x) \mid \phi(y) \rangle|^2.$$

The resulting kernel matrix can be supplied to a classical kernel method. Kernel evaluation and QSVM training are deliberately not performed in this notebook.

## Why is QSVM hybrid?

A QSVM is hybrid because the feature representation or kernel evaluation uses quantum circuits, while the support-vector optimization is a classical machine-learning procedure. The intended architecture is: classical features -> training-only feature reduction -> quantum feature map -> quantum kernel -> classical QSVC.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIRECTORY = PROJECT_ROOT / 'src'
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from data import load_dataset
from preprocessing import prepare_baseline_data
from quantum_features import reduce_to_quantum_features

## Fit the reduction on training data only

The baseline preprocessing first creates the fixed stratified split and fits the scaler on training features. The quantum selector then computes ANOVA F-scores using only the scaled training matrix and training labels.

In [ ]:
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
dataset = load_dataset(DATASET_PATH)
prepared = prepare_baseline_data(dataset)
selector, X_train_quantum, X_test_quantum = reduce_to_quantum_features(
    prepared.X_train_transformed,
    prepared.split.y_train,  # type: ignore[arg-type]
    prepared.X_test_transformed,
)

print('selected_features:', selector.selected_feature_names)
print('train_shape:', X_train_quantum.shape)
print('test_shape:', X_test_quantum.shape)

In [ ]:
assert X_train_quantum.shape == (len(prepared.split.X_train), 4)
assert X_test_quantum.shape == (len(prepared.split.X_test), 4)
assert list(X_train_quantum.columns) == list(X_test_quantum.columns)
print('Sanity check passed: four selected features are ready for a later four-qubit experiment.')

## Selection rule

`TrainingFeatureSelector` ranks every non-target baseline feature by its one-way ANOVA F-score against the training labels. The four highest-scoring features are retained. Exact ties are resolved by the original feature-column order. This makes the representation deterministic and ensures that test data is used only after selection, during transformation.

QSVM, quantum-kernel evaluation, simulated noise, and model metrics are intentionally outside this notebook.